# Lesson 9: Building the Pipeline - Writing Data & Optimization

---

## The Final Report Analogy

Imagine spending two weeks analysing the property market, only to present your findings
by reading them aloud from memory. That's what an analysis that only lives in Spark's
memory is worth — nothing, the moment the cluster shuts down.

In this lesson you'll learn how to **deliver and reuse** your work:

- Write DataFrames to CSV, JSON, and Parquet
- Understand write modes (`overwrite`, `append`, `ignore`, `error`)
- Partition output so downstream readers are fast
- Cache hot DataFrames to avoid recomputation
- Tune shuffle partitions and join strategies
- Wrap everything in a **production pipeline function**

### What you'll learn
| Topic | Key concepts |
|---|---|
| Writing data | `write.csv`, `write.json`, `write.parquet` |
| Write modes | `overwrite`, `append`, `ignore`, `error` |
| Partitioning | `partitionBy`, partition pruning |
| Caching | `cache()`, `persist()`, `unpersist()` |
| Optimization | `repartition`, `coalesce`, `broadcast` |
| Pipeline | End-to-end ETL function |


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, avg, count, sum as spark_sum,
    to_date, datediff, current_date, year, month,
    broadcast, lit
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, DateType
)
from pyspark.storagelevel import StorageLevel

spark = SparkSession.builder \
    .appName("Lesson09_BuildingThePipeline") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Load all datasets
listings = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("data/property_listings.csv")

commissions = spark.read.json("data/agent_commissions.json")

neighborhood_stats = spark.read.parquet("data/neighborhood_stats.parquet")

mortgage_rates = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("data/mortgage_rates.csv")

print(f"Listings:           {listings.count():,} records")
print(f"Commissions:        {commissions.count():,} records")
print(f"Neighborhood stats: {neighborhood_stats.count():,} records")
print(f"Mortgage rates:     {mortgage_rates.count():,} records")
print(f"\nShuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

## Writing Data - Delivering the Report

PySpark can write DataFrames to many formats. The general pattern is:

```python
df.write.mode("overwrite").format("parquet").save("path/")
# OR the shorthand API:
df.write.mode("overwrite").parquet("path/")
```

Spark writes **one file per partition**, so the output is a **directory**, not a single file.
This is by design — it enables parallel writes and reads across a cluster.


In [ ]:
# Write listings to CSV
# header=True ensures the column names are written in the first row
# mode="overwrite" deletes any existing output at that path before writing

listings.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("output/listings_report")

print("Written to: output/listings_report/")
print()
print("The output directory contains:")
print("  part-00000-*.csv  <- actual data (one file per partition)")
print("  _SUCCESS           <- empty marker file written when job completes")
print("  .part-*.crc        <- checksum files (can be ignored)")
print()
print("To read it back:")
print('  spark.read.option("header", "true").csv("output/listings_report")')

# Verify by reading back
verify_csv = spark.read.option("header", "true").csv("output/listings_report")
print(f"\nVerification: read back {verify_csv.count():,} rows from CSV output")

In [ ]:
# Write to JSON (newline-delimited JSON, one JSON object per line)
listings.write \
    .mode("overwrite") \
    .json("output/listings_json")

print("Written to: output/listings_json/")
print()
print("JSON output format is NDJSON (Newline Delimited JSON):")
print('  {"property_id":"P001","address":"123 Main St",...}')
print('  {"property_id":"P002","address":"456 Oak Ave",...}')
print()
print("Good for: REST APIs, streaming pipelines, human-readable debugging")
print("Not ideal for: analytics (no columnar compression, no schema enforcement)")

verify_json = spark.read.json("output/listings_json")
print(f"\nVerification: read back {verify_json.count():,} rows from JSON output")

In [ ]:
# Write to Parquet - the gold standard for analytics workloads
listings.write \
    .mode("overwrite") \
    .parquet("output/listings_parquet")

print("Written to: output/listings_parquet/")
print()
print("Why Parquet is the best format for analytics:")
print()
print("  1. COLUMNAR STORAGE")
print("     Rows are stored by column, not by row.")
print("     Reading only 'list_price' reads ONLY that column from disk.")
print("     CSV must read every column in every row, even ones you don't need.")
print()
print("  2. SCHEMA EMBEDDED")
print("     The file stores the column names AND data types.")
print("     No more inferSchema guessing games.")
print()
print("  3. COMPRESSION")
print("     Columnar layout enables very effective compression (Snappy by default).")
print("     A 100 MB CSV often becomes a 20-30 MB Parquet file.")
print()
print("  4. STATISTICS")
print("     Parquet stores min/max per column per row group.")
print("     Spark can skip entire chunks of the file based on filter conditions.")

verify_parquet = spark.read.parquet("output/listings_parquet")
print(f"\nVerification: read back {verify_parquet.count():,} rows from Parquet output")
print("\nParquet schema (embedded in file — no inferSchema needed):")
verify_parquet.printSchema()

## Write Modes Explained

The `.mode()` option controls what Spark does if the output path already exists.

| Mode | Behavior | Use when |
|---|---|---|
| `overwrite` | Delete existing data, write fresh | Idempotent batch jobs, daily reports |
| `append` | Add new files alongside existing ones | Log ingestion, incremental loads |
| `ignore` | Do nothing if path exists | First-run initialization, idempotent setup |
| `error` | Raise an exception if path exists (default) | Development — prevents accidental overwrites |

### Common mistake
Using `append` in a loop without deduplication causes duplicate data. Always track
what has already been written, or use `overwrite` for full refreshes.

```python
# Safe default for scheduled batch jobs:
df.write.mode("overwrite").parquet("output/daily_report")

# For incremental ingestion:
new_records.write.mode("append").parquet("output/event_log")
```


## Partitioning - Organizing for Speed

When you write with `partitionBy("column")`, Spark creates a **subdirectory per unique value**:

```
output/listings_by_neighborhood/
  neighborhood=Downtown/
    part-00000.parquet
  neighborhood=Suburbs/
    part-00000.parquet
  ...
```

When you later filter on `neighborhood == "Downtown"`, Spark reads **only that subdirectory**.
This is called **partition pruning** and it can reduce I/O by 90%+ on large datasets.

**Choose partition columns wisely:**
- Good: low-to-medium cardinality (neighborhood, year, status)
- Bad: high cardinality (property_id, price) — creates thousands of tiny files


In [ ]:
# Write listings partitioned by neighborhood
# Each neighborhood gets its own subdirectory
listings.write \
    .mode("overwrite") \
    .partitionBy("neighborhood") \
    .parquet("output/listings_by_neighborhood")

print("Written to: output/listings_by_neighborhood/")
print()
print("On disk the structure looks like:")
print("  output/listings_by_neighborhood/")
print("    neighborhood=Downtown/")
print("      part-00000-abc123.snappy.parquet")
print("    neighborhood=Midtown/")
print("      part-00000-def456.snappy.parquet")
print("    neighborhood=Suburbs/")
print("      part-00000-ghi789.snappy.parquet")
print("    ... (one directory per unique neighborhood value)")
print()

# Verify how many partitions were created
distinct_neighborhoods = listings.select("neighborhood").distinct().count()
print(f"Number of partition directories created: {distinct_neighborhoods}")

In [ ]:
# Read back with a neighborhood filter to demonstrate partition pruning
# Spark will scan ONLY the matching subdirectory, not the entire dataset

# First, discover what neighborhoods exist
neighborhoods = [row[0] for row in listings.select("neighborhood").distinct().collect()]
target_neighborhood = neighborhoods[0]  # Pick the first one as our example
print(f"Filtering for neighborhood: '{target_neighborhood}'")

# Read with filter — Spark uses partition pruning
downtown_only = spark.read \
    .parquet("output/listings_by_neighborhood") \
    .filter(col("neighborhood") == target_neighborhood)

print(f"Records in '{target_neighborhood}': {downtown_only.count()}")
print()
print("What happened under the hood:")
print(f"  - Spark saw the filter: neighborhood == '{target_neighborhood}'")
print(f"  - It scanned ONLY: output/listings_by_neighborhood/neighborhood={target_neighborhood}/")
print(f"  - All other neighborhood directories were SKIPPED entirely")
print()
print("On a dataset with 100 neighborhoods, this reads 1/100th of the data!")
print()
downtown_only.select(
    "property_id", "address", "list_price", "property_type", "status"
).show(5, truncate=False)

## Caching - Remember This Data!

By default, Spark recomputes a DataFrame from scratch every time an action is called.
If you call `.count()`, then `.show()`, then `.write()` on the same DataFrame,
Spark reads the source file and runs all transformations **three times**.

**`cache()`** stores the DataFrame in memory (or memory+disk) after the first action.
All subsequent actions reuse the cached data.

```python
df.cache()      # shorthand for persist(StorageLevel.MEMORY_AND_DISK)
df.count()      # triggers the first computation — data is now cached
df.show()       # reads from cache — fast!
df.unpersist()  # release memory when done
```

**When to cache:**
- A DataFrame is used in multiple downstream actions
- The DataFrame is expensive to compute (many joins, aggregations)

**When NOT to cache:**
- The DataFrame is only used once
- The DataFrame is larger than available executor memory


In [ ]:
import time

# Build an enriched DataFrame (simulate an expensive transformation chain)
enriched = listings \
    .withColumn("property_age", lit(2024) - col("year_built")) \
    .withColumn(
        "price_category",
        when(col("list_price") < 300000, "Budget")
        .when(col("list_price") < 600000, "Mid-Range")
        .when(col("list_price") < 1000000, "Premium")
        .otherwise("Luxury")
    ) \
    .withColumn(
        "age_category",
        when(col("property_age") < 5, "Brand New")
        .when(col("property_age") < 20, "Modern")
        .when(col("property_age") < 40, "Established")
        .otherwise("Vintage")
    )

# WITHOUT caching: each action re-reads and re-transforms from scratch
t0 = time.time()
count1 = enriched.count()
t1 = time.time()
count2 = enriched.filter(col("price_category") == "Luxury").count()
t2 = time.time()
print(f"Without cache:")
print(f"  First action (count):          {t1 - t0:.3f}s")
print(f"  Second action (luxury filter): {t2 - t1:.3f}s")

# WITH caching
enriched.cache()

t3 = time.time()
count3 = enriched.count()  # triggers caching — data is materialized
t4 = time.time()
count4 = enriched.filter(col("price_category") == "Luxury").count()
t5 = time.time()
print(f"\nWith cache:")
print(f"  First action (fills cache):    {t4 - t3:.3f}s")
print(f"  Second action (from cache):    {t5 - t4:.3f}s  <- faster!")

# Always release memory when you're done with a cached DataFrame
enriched.unpersist()
print(f"\nCache released. Total rows: {count1:,}, Luxury listings: {count4:,}")

## Optimizing Joins - Broadcast & Shuffle

Joins are the most expensive operation in distributed computing. Every row from the left
table must be matched with a row from the right table — across the network if needed.

### Broadcast Join (small table)
If one table is small enough to fit in memory on every executor (typically < 10 MB),
Spark can **broadcast** it — send a full copy to every node. No shuffle needed.

```python
# Force a broadcast join:
result = large_df.join(broadcast(small_df), "key")
```

### `repartition` vs `coalesce`

| Operation | Behavior | Use when |
|---|---|---|
| `repartition(n)` | Full shuffle, balanced partitions | Before a wide aggregation or join |
| `coalesce(n)` | No shuffle, merges existing partitions | Reducing partitions before writing |


In [ ]:
# Demonstrate repartition vs coalesce
print(f"Original partition count: {listings.rdd.getNumPartitions()}")

# repartition: full shuffle, perfectly balanced across N partitions
repartitioned = listings.repartition(8)
print(f"After repartition(8):     {repartitioned.rdd.getNumPartitions()} partitions")
print("  -> Full shuffle: all data moves across the network")
print("  -> Good before a join or aggregation that benefits from more parallelism")

# coalesce: no shuffle, just merges nearby partitions
coalesced = listings.coalesce(2)
print(f"\nAfter coalesce(2):        {coalesced.rdd.getNumPartitions()} partitions")
print("  -> No shuffle: data stays local, partitions are merged")
print("  -> Good before writing: reduces the number of output files")
print("  -> Cannot INCREASE partition count (use repartition for that)")

print()
print("Broadcast join demo (neighborhood_stats is tiny — 8 rows):")
joined = listings.join(
    broadcast(neighborhood_stats),
    on="neighborhood",
    how="left"
)
print(f"Joined dataset: {joined.count():,} rows")
print("  -> broadcast() hint tells Spark to send neighborhood_stats to every executor")
print("  -> No shuffle of the listings table — massive performance gain!")
joined.select("property_id", "neighborhood", "list_price", "school_rating", "crime_rate_per_1000").show(5)

In [ ]:
# Production pipeline: wrap the entire ETL flow in a function
# This makes it easy to schedule, test, and reuse

def run_market_analysis_pipeline(spark, input_path="data", output_path="output"):
    """
    End-to-end market analysis ETL pipeline.

    Steps:
      1. Ingest all source datasets
      2. Clean and enrich listings
      3. Join with neighborhood statistics
      4. Aggregate commission data per property
      5. Write partitioned output
    """
    print("=" * 60)
    print(" MARKET ANALYSIS PIPELINE")
    print("=" * 60)

    # --- STEP 1: INGEST ---
    print("\n[1/5] Ingesting source data...")
    listings_raw = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(f"{input_path}/property_listings.csv")

    commissions_raw = spark.read.json(f"{input_path}/agent_commissions.json")
    neighborhood_raw = spark.read.parquet(f"{input_path}/neighborhood_stats.parquet")

    print(f"  Listings:     {listings_raw.count():,}")
    print(f"  Commissions:  {commissions_raw.count():,}")
    print(f"  Neighborhoods:{neighborhood_raw.count():,}")

    # --- STEP 2: CLEAN & ENRICH ---
    print("\n[2/5] Cleaning and enriching listings...")
    from pyspark.sql.functions import trim, upper

    enriched = listings_raw \
        .withColumn("property_type", upper(trim(col("property_type")))) \
        .withColumn("neighborhood", trim(col("neighborhood"))) \
        .withColumn("property_age", lit(2024) - col("year_built")) \
        .withColumn("price_per_sqft", col("list_price") / col("sqft")) \
        .withColumn(
            "price_category",
            when(col("list_price") < 300000, "Budget")
            .when(col("list_price") < 600000, "Mid-Range")
            .when(col("list_price") < 1000000, "Premium")
            .otherwise("Luxury")
        ) \
        .withColumn(
            "days_on_market",
            datediff(current_date(), to_date(col("listing_date"), "yyyy-MM-dd"))
        )

    print(f"  Enriched columns added: property_age, price_per_sqft, price_category, days_on_market")

    # --- STEP 3: JOIN NEIGHBORHOOD STATS ---
    print("\n[3/5] Joining neighborhood statistics...")
    master = enriched.join(broadcast(neighborhood_raw), on="neighborhood", how="left")
    print(f"  Master dataset rows: {master.count():,}")

    # --- STEP 4: AGGREGATE COMMISSIONS ---
    print("\n[4/5] Aggregating commission data...")
    commission_agg = commissions_raw.groupBy("property_id").agg(
        spark_sum("commission_amount").alias("total_commission"),
        avg("commission_rate").alias("avg_commission_rate")
    )
    master_with_commissions = master.join(commission_agg, on="property_id", how="left")
    print(f"  Master with commissions rows: {master_with_commissions.count():,}")

    # Cache before writing multiple outputs
    master_with_commissions.cache()

    # --- STEP 5: WRITE OUTPUT ---
    print("\n[5/5] Writing output...")

    master_with_commissions \
        .write.mode("overwrite") \
        .partitionBy("neighborhood") \
        .parquet(f"{output_path}/market_analysis_master")
    print(f"  Written: {output_path}/market_analysis_master/ (partitioned by neighborhood)")

    master_with_commissions \
        .filter(col("status") == "Active") \
        .coalesce(1) \
        .write.mode("overwrite") \
        .option("header", "true") \
        .csv(f"{output_path}/active_listings_report")
    print(f"  Written: {output_path}/active_listings_report/ (active only, single CSV file)")

    master_with_commissions.unpersist()

    print("\n" + "=" * 60)
    print(" PIPELINE COMPLETE")
    print("=" * 60)
    return master_with_commissions


# Run the pipeline!
result = run_market_analysis_pipeline(spark)

## Lesson 9 Complete!

---

### What you mastered today

| Skill | Key takeaway |
|---|---|
| Writing CSV | Always set `header=True`; output is a directory, not a file |
| Writing JSON | NDJSON format; good for APIs, not ideal for analytics |
| Writing Parquet | Best for analytics: columnar, compressed, schema-embedded |
| Write modes | `overwrite` for batch jobs; `append` for incremental loads |
| Partitioned writes | `partitionBy()` enables partition pruning on reads |
| Caching | `cache()` + first action materializes; always `unpersist()` |
| `repartition` vs `coalesce` | `repartition` = full shuffle; `coalesce` = no shuffle |
| Broadcast joins | `broadcast(small_df)` eliminates shuffle of the large table |
| Pipeline function | Wrap ETL logic in a function for reuse and scheduling |

### Production checklist for any Spark job

- [ ] Set `spark.sql.shuffle.partitions` to match your data size (default 200 is too high for small data)
- [ ] Use Parquet for intermediate and final outputs
- [ ] `partitionBy()` on columns you frequently filter on
- [ ] `broadcast()` any lookup table smaller than ~10 MB
- [ ] `cache()` DataFrames used in multiple actions; always `unpersist()` when done
- [ ] `coalesce(1)` before writing a report that needs to be a single file

---

### Up Next: Lesson 10 - The Grand Market Analysis (Capstone!)

You've spent 9 lessons building up your Spark toolkit. In the final lesson
you'll bring **everything together** in a real-world, end-to-end analytics project.
The board wants a complete market report — and you have exactly the skills to deliver it.

> *"A pipeline that runs once and stays in memory is a prototype.
> A pipeline that writes to disk, handles failures, and can be re-run tomorrow is production."*
